# Clase 05 — Distribuciones de probabilidad

**Curso:** Estadística Aplicada con Python · **Bloque 2:** Probabilidad
**Tiempo estimado:** 105 minutos · **Capítulo del manual:** `manual/05_distribuciones_probabilidad.md`

## Qué vas a ver en esta clase

- **Variables aleatorias**, función de masa (pmf), densidad (pdf), función de distribución acumulada (cdf) y cuantiles (ppf).
- Distribuciones como **modelos de los datos**: binomial, Poisson, normal y exponencial, con `scipy.stats`.
- La **densidad no es una probabilidad**; el **z-score** y la regla 68-95-99,7.
- La **t de Student**: la distribución de un *estadístico*, no de los datos.
- El **teorema del límite central** por simulación, y un caso donde falla (Cauchy).

## Requisitos

- Clases 2 a 4 (z-score, ECDF e histogramas, probabilidad y simulación).
- Librerías: `numpy`, `pandas`, `matplotlib`, `scipy` (`pip install numpy pandas matplotlib scipy`; ya vienen instaladas en Google Colab).

> **Cómo usar este notebook:** ejecuta las celdas en orden. Antes de cada experimento escribe qué esperas que ocurra; después compara con lo que ves. Todos los números de los ejemplos son un *ejemplo ilustrativo*, no datos reales.

---
## Celda 0 — Setup

Importamos las librerías y fijamos la **semilla** del generador aleatorio: con la misma semilla, todos obtenemos los mismos números.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerías cargadas correctamente.")

---
## 1. Variable discreta: la binomial

10 clientes visitan una tienda y cada uno compra con probabilidad 0,3 (*ejemplo ilustrativo*). Sea X = número de compradores. Con `stats.binom(n, p)` obtenemos la **pmf** (probabilidad de cada valor) y la **cdf** (probabilidad acumulada).

In [ ]:
compras = stats.binom(n=10, p=0.3)
k = np.arange(0, 11)

tabla = pd.DataFrame({"k": k,
                      "pmf: P(X = k)": compras.pmf(k),
                      "cdf: P(X <= k)": compras.cdf(k)}).set_index("k")
display(tabla.round(4))
print("Suma de la pmf:", compras.pmf(k).sum().round(6))
print("P(X >= 5) =", compras.sf(4).round(4), "  cuantil 0,9:", compras.ppf(0.9))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(k, compras.pmf(k)); ax[0].set_title("Función de masa (pmf)")
ax[1].step(k, compras.cdf(k), where="post"); ax[1].set_title("Función acumulada (cdf): una escalera")
for a in ax: a.set_xlabel("k")
plt.show()

**Experimento guiado 1 — ¿la simulación coincide con la pmf?** Simulamos 100 000 tiendas con 10 clientes cada una y comparamos la frecuencia relativa de cada valor con la pmf teórica.

In [ ]:
sim = compras.rvs(size=100_000, random_state=rng)
frec = np.bincount(sim, minlength=11) / len(sim)

comp = pd.DataFrame({"teórica": compras.pmf(k), "simulada": frec}, index=k)
comp["diferencia"] = (comp["simulada"] - comp["teórica"]).abs()
display(comp.round(4))

# tolerancia: 5 errores típicos de una proporción (peor caso p = 0,5)
tol = 5 * np.sqrt(0.25 / len(sim))
assert comp["diferencia"].max() < tol
print("Media simulada:", sim.mean().round(3), "  teórica:", compras.mean(), "  varianza simulada:", sim.var().round(3), "  teórica:", round(compras.var(), 3))

**¿Qué observas?**

- ¿Qué tan cerca están las frecuencias simuladas de la pmf? ¿Dónde es mayor la diferencia?
- La media y la varianza simuladas, ¿coinciden con n·p y n·p·(1 − p)?
- ¿Qué pasaría con la forma de la pmf si p fuera 0,05 en vez de 0,3?

*(Tu respuesta aquí.)*

---
## 2. Conteos: Poisson y sobredispersión

Un servicio de soporte recibe de media 4 tickets por hora (*ejemplo ilustrativo*). La Poisson tiene un rasgo distintivo: **media = varianza**. Vamos a comparar datos que cumplen esa igualdad con datos de conteo con **sobredispersión** (misma media, mucha más varianza).

In [ ]:
tickets = stats.poisson(mu=4)
print("P(X = 6) =", tickets.pmf(6).round(4), "  P(X <= 2) =", tickets.cdf(2).round(4), "  P(X = 0) =", tickets.pmf(0).round(4))

n_horas = 20_000
datos_poisson = tickets.rvs(size=n_horas, random_state=rng)
# mismos 4 tickets de media, pero con ritmo variable (binomial negativa: varianza = 12)
datos_sobre = rng.negative_binomial(n=2, p=2 / 6, size=n_horas)

resumen = pd.DataFrame({
    "media": [datos_poisson.mean(), datos_sobre.mean()],
    "varianza": [datos_poisson.var(ddof=1), datos_sobre.var(ddof=1)],
}, index=["Poisson", "sobredispersión"])
resumen["var / media"] = resumen["varianza"] / resumen["media"]
display(resumen.round(3))

kk = np.arange(0, 21)
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(kk - 0.2, np.bincount(datos_poisson, minlength=21)[:21] / n_horas, width=0.4, label="datos Poisson")
ax.bar(kk + 0.2, np.bincount(datos_sobre, minlength=21)[:21] / n_horas, width=0.4, label="datos sobredispersos")
ax.plot(kk, tickets.pmf(kk), "ko-", label="modelo Poisson (λ = 4)")
ax.set_xlabel("tickets por hora"); ax.legend(); ax.set_title("Poisson frente a datos con sobredispersión")
plt.show()

print("P(X >= 12): modelo Poisson =", tickets.sf(11).round(5), " datos sobredispersos =", (datos_sobre >= 12).mean().round(5))

**¿Qué observas?**

- ¿Cuál es el cociente varianza / media en cada conjunto? ¿Cuál es compatible con Poisson?
- ¿Qué probabilidad de ≥ 12 tickets predice el modelo Poisson y cuánta hay realmente en los datos sobredispersos?
- ¿Qué consecuencia práctica tendría dimensionar el equipo con el modelo equivocado?

*(Tu respuesta aquí.)*

---
## 3. Variable continua: densidad, área y z-score

Alturas con media 170 cm y desviación estándar 7 cm (*ejemplo ilustrativo*). Tres ideas: (1) la **probabilidad es un área**, (2) la **densidad puede ser mayor que 1**, (3) el **z-score** traduce un valor a un percentil.

In [ ]:
altura = stats.norm(loc=170, scale=7)

# 1) probabilidad = área = diferencia de la cdf
print("P(X < 180) =", altura.cdf(180).round(4), "  P(160 < X < 180) =", (altura.cdf(180) - altura.cdf(160)).round(4))
print("P(X = 170) = 0   pero la densidad en 170 es", altura.pdf(170).round(4))

# 2) densidad mayor que 1 (y aun así el área total es 1)
estrecha = stats.norm(loc=0, scale=0.1)
xs = np.linspace(-1, 1, 20_001)
print("Densidad máxima de la normal estrecha:", estrecha.pdf(0).round(3),
      " área total ≈", (estrecha.pdf(xs).sum() * (xs[1] - xs[0])).round(4))

# 3) z-score y percentil
z = (180 - 170) / 7
print("z =", round(z, 3), "  percentil:", (100 * stats.norm.cdf(z)).round(1), "%   percentil 97,5 →", altura.ppf(0.975).round(2), "cm")

xs2 = np.linspace(145, 195, 500)
plt.plot(xs2, altura.pdf(xs2))
zona = (xs2 >= 160) & (xs2 <= 180)
plt.fill_between(xs2[zona], altura.pdf(xs2[zona]), alpha=0.4, label="P(160 < X < 180)")
plt.legend(); plt.xlabel("cm"); plt.ylabel("densidad"); plt.title("La probabilidad es el área bajo la curva")
plt.show()

### La regla 68-95-99,7

Comprobamos la regla con la cdf y con una muestra simulada de 100 000 alturas.

In [ ]:
muestra = altura.rvs(size=100_000, random_state=rng)
filas = []
for kσ in (1, 2, 3):
    exacta = stats.norm.cdf(kσ) - stats.norm.cdf(-kσ)
    empirica = (np.abs(muestra - 170) < kσ * 7).mean()
    filas.append({"intervalo": f"μ ± {kσ}σ", "exacta": exacta, "simulada": empirica})
display(pd.DataFrame(filas).set_index("intervalo").round(4))

**¿Qué observas?** ¿Coinciden la proporción exacta y la simulada? Ahora piensa en los sueldos de la clase 2 (muy asimétricos): ¿tendría sentido decir que "el 95 % de los sueldos cae a menos de 2 desviaciones de la media"?

*(Tu respuesta aquí.)*

---
## 4. Tiempos de espera: la exponencial

El tiempo hasta la próxima llamada tiene media 5 minutos (*ejemplo ilustrativo*). Es asimétrica y **no tiene memoria**.

In [ ]:
espera = stats.expon(scale=5)                      # scale = media = 1/λ
tiempos = espera.rvs(size=100_000, random_state=rng)

print("P(T > 10) =", espera.sf(10).round(4), "  simulada:", (tiempos > 10).mean().round(4))
print("mediana teórica:", espera.ppf(0.5).round(3), "  media teórica:", espera.mean())
print("Sin memoria: P(T > 15 | T > 5) =", (espera.sf(15) / espera.sf(5)).round(4), "  simulada:", (tiempos[tiempos > 5] > 15).mean().round(4))

plt.hist(tiempos, bins=60, density=True, alpha=0.5, label="simulación")
xs = np.linspace(0, 30, 300)
plt.plot(xs, espera.pdf(xs), "k", label="pdf")
plt.axvline(espera.mean(), color="red", linestyle="--", label="media"); plt.axvline(espera.ppf(0.5), color="green", linestyle="--", label="mediana")
plt.legend(); plt.xlabel("minutos"); plt.title("Exponencial: muy asimétrica a la derecha")
plt.show()

**¿Qué observas?** ¿Por qué la mediana es menor que la media? ¿Qué significa que P(T > 15 | T > 5) sea igual a P(T > 10)?

*(Tu respuesta aquí.)*

---
## 5. La t de Student: la distribución de un estadístico

**Ojo:** la t **no** modela los datos. Es la distribución de la media estandarizada con la desviación estándar *estimada* s: T = (x̄ − μ) / (s / √n). Lo comprobamos simulando 200 000 muestras de n = 5 datos de una normal (*ejemplo ilustrativo*) y comparando con la normal estándar y con la t de 4 grados de libertad.

In [ ]:
n = 5
datos = rng.normal(loc=0, scale=1, size=(200_000, n))
T = datos.mean(axis=1) / (datos.std(axis=1, ddof=1) / np.sqrt(n))

print("P(|T| > 2) simulada =", (np.abs(T) > 2).mean().round(4))
print("P(|T| > 2) t con 4 gl   =", (2 * stats.t.sf(2, df=4)).round(4))
print("P(|T| > 2) normal estándar =", (2 * stats.norm.sf(2)).round(4))

filas = [{"grados de libertad": gl, "P(|T| > 2)": 2 * stats.t.sf(2, df=gl)} for gl in (2, 4, 10, 30, 100)]
filas.append({"grados de libertad": "normal", "P(|T| > 2)": 2 * stats.norm.sf(2)})
display(pd.DataFrame(filas).set_index("grados de libertad").round(4))

xs = np.linspace(-6, 6, 500)
plt.hist(T[np.abs(T) < 6], bins=120, density=True, alpha=0.4, label="T simulada")
plt.plot(xs, stats.t.pdf(xs, df=4), label="t (4 gl)")
plt.plot(xs, stats.norm.pdf(xs), "k--", label="normal estándar")
plt.legend(); plt.title("La t tiene colas más pesadas que la normal")
plt.show()

**¿Qué observas?**

- ¿Se parece más la T simulada a la normal o a la t de 4 grados de libertad? ¿Dónde se nota la diferencia: en el centro o en las colas?
- ¿Qué pasa con P(|T| > 2) al aumentar los grados de libertad?
- Recuerda: los datos eran normales y aun así T no es normal. ¿Por qué? (Pista: ¿qué se estima con la muestra?)

*(Tu respuesta aquí.)*

---
## 6. Experimento guiado 2 — el teorema del límite central

Población: **exponencial de media 1**, muy asimétrica (asimetría teórica = 2). Para cada n en {1, 5, 30, 200} repetimos 10 000 veces «tomar n datos y calcular su media». Teoría: la media de las medias es 1, su desviación estándar es 1/√n y su asimetría es 2/√n.

Antes de ejecutar, escribe tu predicción: ¿qué forma tendrá el histograma de las medias con n = 200?

In [ ]:
NS = [1, 5, 30, 200]
REPS = 10_000
medias = {n: rng.exponential(1.0, size=(REPS, n)).mean(axis=1) for n in NS}

filas = []
for n in NS:
    m = medias[n]
    assert abs(m.mean() - 1) < 5 / np.sqrt(n * REPS)          # tolerancia: 5 errores típicos
    filas.append({"n": n, "media de las medias": m.mean(), "desv. estándar": m.std(),
                  "1/√n": 1 / np.sqrt(n), "asimetría": stats.skew(m), "2/√n": 2 / np.sqrt(n)})
display(pd.DataFrame(filas).set_index("n").round(3))

fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for ax, n in zip(axes, NS):
    m = medias[n]
    ax.hist(m, bins=40, density=True, alpha=0.6)
    xs = np.linspace(m.min(), m.max(), 300)
    ax.plot(xs, stats.norm(1, 1 / np.sqrt(n)).pdf(xs), "k", lw=1.5)
    ax.set_title(f"n = {n}\nasimetría = {stats.skew(m):.2f}")
plt.suptitle("Medias de n datos exponenciales, con la curva normal del TLC superpuesta", y=1.03)
plt.show()

**¿Qué observas?**

- ¿Qué forma tiene el histograma con n = 1? ¿Y con n = 200? ¿En qué n empieza a parecerse a la curva negra?
- ¿Cómo cambia la anchura (desviación estándar)? ¿Y la asimetría? ¿Con n = 30 está ya cerca de 0?
- Cada dato individual sigue siendo exponencial. ¿Qué es lo que se vuelve aproximadamente normal: los datos o la media?

*(Tu respuesta aquí.)*

---
## 7. Cuando el TLC falla: la distribución de Cauchy

La Cauchy tiene forma de campana pero **no tiene media ni varianza finitas** (colas muy pesadas). El TLC exige varianza finita. Comparamos, a medida que se acumulan datos, la **media acumulada** de una exponencial y la de una Cauchy, y luego el rango intercuartílico de las medias con n = 10 y n = 1000.

In [ ]:
N = 5_000
exp_datos = rng.exponential(1.0, N)
cau_datos = rng.standard_cauchy(N)
pasos = np.arange(1, N + 1)

plt.plot(pasos, np.cumsum(exp_datos) / pasos, label="exponencial (se estabiliza en 1)")
plt.plot(pasos, np.cumsum(cau_datos) / pasos, label="Cauchy (no se estabiliza)")
plt.axhline(1, color="black", linestyle="--")
plt.ylim(-6, 6); plt.xscale("log"); plt.legend(); plt.xlabel("número de datos (escala log)"); plt.ylabel("media acumulada")
plt.title("La media de una Cauchy no converge")
plt.show()

REPS_C = 5_000
filas = []
for nombre, gen in (("exponencial", lambda size: rng.exponential(1.0, size)), ("Cauchy", lambda size: rng.standard_cauchy(size))):
    iqr = {n: stats.iqr(gen((REPS_C, n)).mean(axis=1)) for n in (10, 1000)}
    filas.append({"distribución": nombre, "IQR (n = 10)": iqr[10], "IQR (n = 1000)": iqr[1000], "cociente": iqr[1000] / iqr[10]})
display(pd.DataFrame(filas).set_index("distribución").round(3))

**¿Qué observas?**

- ¿Qué le pasa a la media acumulada de la exponencial y a la de la Cauchy al añadir datos? ¿Hay "saltos" en la Cauchy? ¿Por qué?
- El cociente de rangos intercuartílicos (n = 1000 frente a n = 10): ¿cuánto vale en cada caso? ¿Qué esperarías si el TLC funcionara (1/√100 = 0,1)?
- Si tus datos reales tuvieran valores extremos enormes de vez en cuando, ¿qué precaución tomarías antes de usar la normal para la media?

*(Tu respuesta aquí.)*

---
## Resumen de la clase

- Una **variable aleatoria** asigna números a resultados del azar; la **pmf** (discretas) da P(X = x) y la **pdf** (continuas) da una densidad: la probabilidad es el **área** y P(X = x) = 0.
- La **cdf** acumula probabilidad; la **ppf** (cuantiles) es su inversa. En `scipy.stats`: `pmf`/`pdf`, `cdf`, `sf`, `ppf`, `rvs`.
- **Binomial** (éxitos en n ensayos), **Poisson** (conteos; media = varianza, si no, hay sobredispersión), **normal** (campana; 68-95-99,7; z-score) y **exponencial** (esperas) son **modelos de los datos**.
- La **t de Student** es la distribución de un **estadístico**, con colas más pesadas y convergente a la normal.
- El **TLC**: la **media** de n datos independientes con varianza finita es aproximadamente normal, con desviación σ/√n; es una aproximación, con asimetría fuerte n = 30 puede no bastar y con Cauchy no funciona.
- Siguiente clase: **estimación e intervalos de confianza** (clase 6).